In [ ]:
import geopandas as gpd

from paths import paths, prepare_preprocessing

In [ ]:
prepare_preprocessing()

In [ ]:
lsoa_base = gpd.read_parquet(paths.preprocessing.shared.lsoa)

In [ ]:
def combine_components(minutes):
    files = paths.preprocessing.walkability[minutes]

    amenities = gpd.read_parquet(files.amenities).drop(columns = 'geometry')
    dwellings = gpd.read_parquet(files.dwellings).drop(columns = 'geometry')
    land_use = gpd.read_parquet(files.land_use).drop(columns = 'geometry')
    streets = gpd.read_parquet(files.streets).drop(columns = 'geometry')
    transport = gpd.read_parquet(files.transport).drop(columns = 'geometry')

    return (
        lsoa_base.merge(amenities, on = key, how = 'outer')
        .merge(dwellings, on = key, how = 'outer')
        .merge(land_use, on = key, how = 'outer')
        .merge(streets, on = key, how = 'outer')
        .merge(transport, on = key, how = 'outer')
    )

In [ ]:
# Calculate ranks and percentiles for each metric, and merge into single dataframe
metrics = ['amenity', 'dwelling', 'land_use', 'street', 'transport']
key = 'LSOA21CD'

amenity_cols = [
    'amenity_count_essentials',
    'amenity_count_eating_out',
    'amenity_count_education',
    'amenity_count_healthcare',
    'amenity_count_culture',
    'amenity_count_leisure',
    'amenity_count_services'
]

dwelling_cols = [
    'dwelling_fraction_mid_terrace',
    'dwelling_fraction_semi_detached',
    'dwelling_fraction_end_terrace',
    'dwelling_fraction_detached',
    'dwelling_fraction_enclosed_end_terrace',
    'dwelling_fraction_enclosed_mid_terrace'
]

land_use_cols = [
    'land_use_fraction_green_space',
    'land_use_fraction_blue_space',
    'land_use_fraction_no_data',
    'land_use_fraction_disturbance',
    'land_use_fraction_residential',
    'land_use_fraction_industrial'
]

street_cols = [
    'street_intersection_density',
    'street_length_density',
    'street_intersection_degree_avg',
    'street_circuity_avg',
    'street_dead_end_density',
    'street_betweenness_avg',
    'street_block_length_avg'
]

transport_cols = [
    'transport_stop_count_bus',
    'transport_stop_count_rail',
    'transport_stop_count_metro',
    'transport_stop_count_tram',
    'transport_stop_count_boat',
    'transport_stop_count_light_rail'
]

slim_cols = [
    'LSOA21CD',
    'LSOA21NM',
    'geometry',
    'centroid',
    *[f'{m}_score' for m in metrics],
    *[f'{m}_rank' for m in metrics],
    *[f'{m}_percentile' for m in metrics]
]

enriched_cols = [
    *slim_cols,
    *amenity_cols,
    *dwelling_cols,
    *land_use_cols,
    *street_cols,
    *transport_cols
]

In [ ]:
def add_walkability_score(gdf, metrics):
    metric_weights = {m: 1 for m in metrics}

    z_cols = []
    active_wts = []

    for m in metrics:
        m_score = f'{m}_score'
        w = metric_weights.get(m, 0)

        mean = gdf[m_score].mean()
        std = gdf[m_score].std(ddof = 0)
        gdf[m_score] = (gdf[m_score] - mean) / std

        gdf[m_score] = gdf[m_score] * w

        if w > 0:
            z_cols.append(m_score)
            active_wts.append(w)

    gdf['walkability_score'] = gdf[z_cols].sum(axis = 1) / sum(active_wts)
    gdf['walkability_rank'] = gdf['walkability_score'].rank(ascending = False, method = 'min')
    gdf['walkability_percentile'] = gdf['walkability_score'].rank(ascending = True, method = 'min') / len(gdf)

    for m in metrics:
        m_score = f'{m}_score'
        gdf[f'{m}_rank'] = gdf[m_score].rank(ascending = False, method = 'min')

    return gdf

In [ ]:
for minutes in (10, 15, 20):
    files = paths.preprocessing.walkability[minutes]
    data = combine_components(minutes)

    for col in metrics:
        data[f'{col}_rank'] = data[f'{col}_score'].rank(method = 'min', ascending = False)
        data[f'{col}_percentile'] = data[f'{col}_score'].rank(ascending = True, method = 'min') / len(data)
    
    slim = data[slim_cols].copy()
    enriched = data[enriched_cols].copy()
    
    slim = add_walkability_score(slim, metrics)
    enriched = add_walkability_score(enriched, metrics)
    
    slim.to_parquet(files.slim, index = False)
    enriched.to_parquet(files.enriched, index = False)